# Create round_info.csv

Builds `round_info.csv`, the per-round series / HAL config / data folder table
that notebook 06 turns into the Dave recipe, from the HAL configs (01, 04) and
the positions (03).

**Variable z per FOV:** bits rows get `tissue_thickness="multi"` and
`z_lengths` (every tier's frame count, ascending, JSON) from notebook 04's
`metadata/z_tiers.csv`. Their `hal_config` is the deepest tier; the per-FOV
tier comes from the positions file's 3rd column.

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/before_imaging/multi_z/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.dave      import create_round_info, create_round_info_multitissue
from MERci.acquisition.positions import (
    discover_boundary_files, resolve_boundaries_source_dir, group_boundaries_by_path_mode,
)

In [ ]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
MULTI_Z_DIR   = SAMPLE_DIR / "multi_z"      # tier hal_configs/shutters (notebook 04)
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# Positions files are named with POSITIONS_TAG.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

In [ ]:
# ── Experiment parameters ──────────────────────────────────────
MICROSCOPE  = "ST2"   # microscope identifier

# Must match notebook 03. "legacy"/"union" join a tissue's boundaries into one
# segment; "transit" keeps them separate.
TISSUE_PATH_MODE           = "legacy"
TISSUE_PATH_MODE_OVERRIDES = {}

def tissue_path_mode(t):
    mode = TISSUE_PATH_MODE_OVERRIDES.get(t, TISSUE_PATH_MODE)
    if mode not in ("legacy", "transit", "union"):
        raise ValueError(f"Tissue {t}: path mode must be 'legacy', 'transit' or 'union', got {mode!r}")
    return mode

# ── Detect the tissue/boundary layout (written by notebook 03) ──────
BOUNDARY_SOURCE = None
BOUNDARY_DIR, BOUNDARY_SOURCE = resolve_boundaries_source_dir(POSITIONS_DIR, BOUNDARY_SOURCE)
print(f"BOUNDARY_SOURCE: {BOUNDARY_SOURCE}")
print(f"BOUNDARY_DIR   : {BOUNDARY_DIR}")

boundaries, MODE = discover_boundary_files(BOUNDARY_DIR)
groups           = group_boundaries_by_path_mode(boundaries, MODE, tissue_path_mode)
MULTI_BOUNDARY   = len(groups) > 1
print(f"Layout mode: {MODE}  ({len(boundaries)} boundary file(s) -> {len(groups)} segment(s)) -> "
      f"{'per-segment' if MULTI_BOUNDARY else 'single-positions'} recipe")

# Cells/transit HAL configs from notebook 01 (settings/).
cells_hal_configs   = sorted(SETTINGS_DIR.glob("hal-config-*cells*.xml"))
transit_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*transit*.xml"))
CELLS_HAL_CONFIG   = cells_hal_configs[0].name   if cells_hal_configs   else "hal-config-mf3-cells.xml"
TRANSIT_HAL_CONFIG = transit_hal_configs[0].name if transit_hal_configs else None

# Bits: notebook 04's tiers; the deepest (last row) is round_info's hal_config.
z_tiers_path = METADATA_DIR / "z_tiers.csv"
if not z_tiers_path.exists():
    raise FileNotFoundError(f"{z_tiers_path} not found -- run notebook 04 first.")
z_tiers_df = pd.read_csv(z_tiers_path).sort_values("z_max_um").reset_index(drop=True)

BITS_HAL_CONFIG = z_tiers_df.iloc[-1]["hal_stem"] + ".xml"   # deepest tier, e.g. "...deep-....xml"
Z_LENGTHS_JSON  = json.dumps(sorted(int(n) for n in z_tiers_df["n_frames"]))

print(f"\nCells   HAL config : {CELLS_HAL_CONFIG}")
print(f"Transit HAL config : {TRANSIT_HAL_CONFIG}")
print(f"Bits    HAL config (deepest tier, representative) : {BITS_HAL_CONFIG}")
print(f"z_lengths (all tiers' frame counts)                : {Z_LENGTHS_JSON}")

if MULTI_BOUNDARY and TRANSIT_HAL_CONFIG is None:
    raise FileNotFoundError(
        "Multiple boundaries detected but no hal-config-*transit*.xml in settings/. "
        "Run the transit cell in notebook 01 first."
    )

## Round – bit – color mapping

The round → bit → colour mapping. Sets `N_HYBS` (the highest round) and is
saved to `round_bit_color_map.csv` for notebooks 06 and 07.

In [ ]:
# round : hyb/bit index (1-based) = bits movie series number, not the Dave imaging round
# bit   : bit number     |     color : excitation wavelength (nm)
round_bit_color = [
    (1,  1,  650), (1,  2,  560),
    (2,  3,  560), (2,  4,  650),
    (3,  5,  650), (3,  6,  560),
    (4,  7,  650), (4,  8,  560),
    (5,  9,  560), (5,  10, 650),
    (6,  11, 650), (6,  12, 560),
    (7,  13, 650), (7,  14, 560),
    (8,  15, 560), (8,  16, 650),
    (9,  17, 650), (9,  18, 560),
    (10, 19, 650), (10, 20, 560),
    (11, 21, 560), (11, 22, 650),
    (12, 23, 650), (12, 24, 560),
    (13, 25, 650), (13, 26, 560),
]

rbc_df   = pd.DataFrame(round_bit_color, columns=["round", "bit", "color"])
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
rbc_df.to_csv(rbc_path, index=False)

N_HYBS = int(rbc_df["round"].max())   # number of bits rounds, derived from the mapping
print(f"Saved: {rbc_path}")
print(f"N_HYBS (from mapping): {N_HYBS}")
print(rbc_df.to_string(index=False))

## Build round_info.csv, tagging bits rows as tissue_thickness="multi"

In [ ]:
if MULTI_BOUNDARY:
    round_info = create_round_info_multitissue(
        microscope         = MICROSCOPE,
        n_bits              = N_HYBS,
        bits_hal_config     = BITS_HAL_CONFIG,
        cells_hal_config    = CELLS_HAL_CONFIG,
        transit_hal_config  = TRANSIT_HAL_CONFIG,
        sample_dir          = SAMPLE_DIR,
        boundaries          = boundaries,
        mode                = MODE,
        sample_name         = POSITIONS_TAG,
        tissue_path_mode    = tissue_path_mode,
    )
else:
    round_info = create_round_info(
        microscope       = MICROSCOPE,
        n_bits           = N_HYBS,
        bits_hal_config  = BITS_HAL_CONFIG,
        cells_hal_config = CELLS_HAL_CONFIG,
        sample_dir       = SAMPLE_DIR,
        positions_txt    = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt",
    )

# Bits rows are variable-z rounds; cells/transit rows stay single.
is_bits_row = round_info["imaging_type"] == "bits"
round_info["tissue_thickness"] = None
round_info.loc[is_bits_row, "tissue_thickness"] = "multi"
round_info["z_lengths"] = None
round_info.loc[is_bits_row, "z_lengths"] = Z_LENGTHS_JSON

print(round_info.to_string(index=False))

out_csv = METADATA_DIR / "round_info.csv"
round_info.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")